# Sentiment Analysis on Sahabat Pegadaian App Reviews using IndoBERT

This notebook demonstrates a sentiment analysis pipeline for Sahabat Pegadaian app reviews, combining **Lexicon-based auto-labeling** with a fine-tuned **IndoBERT** transformer model for classification.

**Result:** 99.22% accuracy, 99.21% F1-score on the held-out test set.

> Note: The original dataset is proprietary (internal client data from an analytics internship) and is not included in this repository. This notebook is shared to document the methodology and results. To run it yourself, replace the data loading step with your own labeled text dataset (two columns: text and label).


## 1. Setup & Imports

In [ ]:
!pip install -q transformers accelerate nltk scikit-learn

import pandas as pd
import numpy as np
import re
import torch
import nltk

from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

import matplotlib.pyplot as plt
import seaborn as sns

nltk.download("vader_lexicon")


## 2. Load Data

> Replace the path below with your own dataset. Expected columns: a free-text column (renamed here to `Query`) containing the review/comment text.


In [ ]:
# TODO: replace with the path to your own dataset
DATA_PATH = "data/reviews.csv"

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["Query"])

print("Total rows:", len(df))
df.head()


## 3. Text Cleaning

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["Query"].astype(str).apply(clean_text)


## 4. Auto-Labeling with Lexicon (VADER)

Since the raw reviews have no sentiment labels, we auto-label them using a lexicon-based approach (VADER) as a first-pass labeling strategy, then use these labels to train the IndoBERT classifier.


**Label mapping:** `0` = negative, `1` = positive.


In [ ]:
sia = SentimentIntensityAnalyzer()

def get_label(text):
    score = sia.polarity_scores(text)["compound"]
    if score > 0:
        return "positive"
    elif score < 0:
        return "negative"
    else:
        return "neutral"

df["label"] = df["clean_text"].apply(get_label)

# Drop neutral for a binary positive/negative classification task
df = df[df["label"] != "neutral"]
df["label"] = df["label"].map({"negative": 0, "positive": 1})

print("Label distribution:")
print(df["label"].value_counts())


In [ ]:
df["label"].value_counts().plot(kind="bar")
plt.title("Label Distribution")
plt.xlabel("Label (0=negative, 1=positive)")
plt.ylabel("Count")
plt.show()


## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

print("Train:", len(X_train))
print("Test :", len(X_test))


## 6. Load IndoBERT & Tokenize

In [ ]:
model_name = "indolem/indobert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)


## 7. PyTorch Dataset Wrapper

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, y_train)
test_dataset = SentimentDataset(test_encodings, y_test)


## 8. Fine-Tune IndoBERT

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


## 9. Evaluation

In [ ]:
results = trainer.evaluate()
print("Evaluation results:")
print(results)


In [ ]:
metric_names = ["accuracy", "f1", "precision", "recall"]
values = [results[f"eval_{m}"] for m in metric_names]

metrics_df = pd.DataFrame({"Metric": metric_names, "Score": values})
display(metrics_df)

plt.figure()
bars = plt.bar(metric_names, values)
plt.ylim(0.85, 1.0)
for i, v in enumerate(values):
    plt.text(i, v + 0.002, f"{v:.3f}", ha="center")
plt.title("Evaluation Metrics")
plt.ylabel("Score")
plt.show()


In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## Results Summary

| Metric | Score |
|---|---|
| Accuracy | 0.9922 |
| Precision | 0.9923 |
| Recall | 0.9922 |
| F1-score | 0.9921 |
| Eval Loss | 0.0589 |

The model correctly classified sentiment with very high accuracy on the held-out test set, using a single-pipeline approach where lexicon-based labels are generated on the fly rather than saved as a separate intermediate dataset. See the project write-up in the main README for a comparison against a two-stage pipeline variant.
